# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mhdaamiryo/fly-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane:** Content Refresh / Search Intelligence ranking  
**Month:** March 2026 (`month=2026-03`)  

This notebook defines the data slice, verifies its source grain and coverage using exactly three contract queries, builds five features, and demonstrates label leakage.

> Before running: accept access to `FlyRank/internship-warehouse` and add a Hugging Face READ token in Colab Secrets as `HF_TOKEN`. Never paste the token into a code cell. The live query outputs will be produced by your authenticated run; no numbers have been invented in this committed file.

## 1. Data contract — five answers

1. **One row means:** in the source fact table, one row is one pseudonymized content item for one client on one reporting date: `report_date × client_hash_id × content_hash_id`. The feature frame later aggregates to one row per client-content pair for March.
2. **Tables:** `fact_content_daily_performance` is the primary source. `dim_content` is available for content metadata joins and `dim_clients` for history/access coverage checks; neither is needed for this first search-only frame.
3. **Time window:** March 1–31, 2026. March is a middle panel month. Features summarize observed March search performance. This is a descriptive review-ranking snapshot after March closes, not yet a production prediction design.
4. **Decision / proxy:** rank content items for human refresh review. For the leakage demonstration only, define a temporary proxy label: March impressions are at least 20% below February impressions for the same client-content pair.
5. **Deliberate exclusion:** `client_hash_id` and `content_hash_id` are context/join keys, never model features. GA4 columns are excluded from this feature frame because their availability is uneven; never interpret zero-filled or NULL GA4 values as true zero engagement.

**Field buckets:** features = the five March aggregate signals below; label/proxy = temporary February-to-March decline; context = client/content IDs and report date; excluded = future/outcome fields and GA4 measures without confirmed availability.

## 2. Connect to the gated warehouse

In [ ]:
%pip -q install duckdb huggingface_hub pandas numpy scikit-learn


In [ ]:
import os
import duckdb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')

if not HF_TOKEN:
    raise RuntimeError('Add your Hugging Face READ token as a Colab Secret named HF_TOKEN.')

con = duckdb.connect()
con.execute(f\"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')\")
REL = 'hf://datasets/FlyRank/internship-warehouse'
DAILY = f\"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')\"
FEB = f\"read_parquet('{REL}/fact_content_daily_performance/month=2026-02/*.parquet')\"
print('Warehouse connection configured for February and March partitions.')


## 3. Three verification queries

These are the three contract-verification queries: (1) prove the fact grain, (2) report March row count/date span, and (3) measure GA4 availability using `IS TRUE`. Each displays its real output when the notebook is executed. Do not use the `_sample` table: it is June 2026, the final month.

### Query 1 — prove the grain

The query returns any duplicate report-date/client/content keys. Zero rows means the stated grain holds.

In [ ]:
q1 = con.sql(f\"\"\"
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS rows_at_key
    FROM {DAILY}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    ORDER BY rows_at_key DESC
    LIMIT 10
\"\"\").df()
print('Duplicate keys (expected: zero rows):')
display(q1)
assert q1.empty, 'Duplicate source grain keys found; investigate before proceeding.'


### Query 2 — March row count and date span

In [ ]:
q2 = con.sql(f\"\"\"
    SELECT COUNT(*) AS row_count,
           COUNT(DISTINCT client_hash_id) AS clients,
           COUNT(DISTINCT content_hash_id) AS content_items,
           MIN(report_date) AS min_report_date,
           MAX(report_date) AS max_report_date
    FROM {DAILY}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
\"\"\").df()
display(q2)


### Query 3 — GA4 availability

Use `IS TRUE` because the availability flag can also be NULL. This reports exactly how many rows survive that condition.

In [ ]:
q3 = con.sql(f\"\"\"
    SELECT COUNT(*) AS march_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS rows_ga4_available,
           COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE) AS rows_not_true,
           COUNT(*) FILTER (WHERE ga4_data_available IS NULL) AS rows_flag_null
    FROM {DAILY}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
\"\"\").df()
display(q3)


## 4. Five-feature frame

The descriptive decision point is **after March 31, 2026**. All five fields are calculated only from March daily observations and aggregated to one client-content row.

| Feature | Meaning | Available when? |
|---|---|---|
| `march_gsc_impressions` | Total March search impressions | After March closes; the month’s observations are complete. |
| `march_gsc_clicks` | Total March search clicks | After March closes; measured in the same March window. |
| `march_avg_position` | Average reported GSC position, excluding zero/no-position values | After March closes; computed from March search observations. |
| `march_days_with_impressions` | Count of March dates with at least one impression | After March closes; derived from daily March rows. |
| `march_content_days_observed` | Number of daily fact records for the content item | After March closes; measures the number of observed reporting dates. |

No GA4 columns are used. The availability check above demonstrates why the flag must be respected. This descriptive feature frame is not a future-outcome model; a later model must choose an earlier decision point and non-overlapping future label window.

In [ ]:
feature_sql = f\"\"\"
    SELECT client_hash_id, content_hash_id,
           SUM(COALESCE(gsc_impressions, 0)) AS march_gsc_impressions,
           SUM(COALESCE(gsc_clicks, 0)) AS march_gsc_clicks,
           AVG(NULLIF(gsc_avg_position, 0)) AS march_avg_position,
           COUNT(DISTINCT CASE WHEN COALESCE(gsc_impressions, 0) > 0 THEN report_date END)
               AS march_days_with_impressions,
           COUNT(*) AS march_content_days_observed
    FROM {DAILY}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
    GROUP BY client_hash_id, content_hash_id
\"\"\"
features = con.sql(feature_sql).df()
feature_cols = ['march_gsc_impressions', 'march_gsc_clicks', 'march_avg_position',
                'march_days_with_impressions', 'march_content_days_observed']
print('Feature frame shape:', features.shape)
display(features[feature_cols].describe().T)
display(features.head())


## 5. Deliberate leakage experiment

**Temporary proxy label:** March impressions are at least 20% below February impressions for the same client-content pair.

**The trap:** deliberately put the label-derived column into the predictor matrix. The classifier can read the answer directly, so its score should jump to 1.00 or very close. This is not valid model performance. Compare against a deliberately weak honest model using only February impressions, then remove the leaked field. This is a teaching demonstration of why the label and anything derived from it must never be a feature.

In [ ]:
feb_mar = con.sql(f\"\"\"
    WITH feb AS (
        SELECT client_hash_id, content_hash_id,
               SUM(COALESCE(gsc_impressions, 0)) AS feb_impressions
        FROM {FEB}
        WHERE report_date >= DATE '2026-02-01' AND report_date < DATE '2026-03-01'
        GROUP BY 1, 2
    ),
    mar AS (
        SELECT client_hash_id, content_hash_id,
               SUM(COALESCE(gsc_impressions, 0)) AS mar_impressions
        FROM {DAILY}
        WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
        GROUP BY 1, 2
    )
    SELECT feb.client_hash_id, feb.content_hash_id,
           feb.feb_impressions, mar.mar_impressions
    FROM feb INNER JOIN mar USING (client_hash_id, content_hash_id)
    WHERE feb.feb_impressions > 0
\"\"\").df()
feb_mar['is_declining_label'] = (feb_mar['mar_impressions'] < 0.8 * feb_mar['feb_impressions']).astype(int)
leak_df = feb_mar.dropna(subset=['feb_impressions', 'mar_impressions']).copy()

X_train, X_test, y_train, y_test = train_test_split(
    leak_df[['feb_impressions']], leak_df['is_declining_label'],
    test_size=0.25, random_state=42, stratify=leak_df['is_declining_label'])
honest = DecisionTreeClassifier(max_depth=2, random_state=42)
honest.fit(X_train, y_train)
honest_score = accuracy_score(y_test, honest.predict(X_test))

X_leaky = leak_df[['feb_impressions']].copy()
X_leaky['LEAK_is_declining_label'] = leak_df['is_declining_label']
Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_leaky, leak_df['is_declining_label'], test_size=0.25,
    random_state=42, stratify=leak_df['is_declining_label'])
leaky = DecisionTreeClassifier(random_state=42)
leaky.fit(Xl_train, yl_train)
leaky_score = accuracy_score(yl_test, leaky.predict(Xl_test))

print(f'Rows in experiment: {len(leak_df):,}')
print(f'Honest feature-only accuracy: {honest_score:.4f}')
print(f'Leaky accuracy (label column included): {leaky_score:.4f}')
print('Leaky feature importance:', dict(zip(X_leaky.columns, leaky.feature_importances_)))


### Remove the leak

The leaky score is intentionally invalid. Drop the label-derived field and preserve only the honest feature list.

In [ ]:
X_leaky = X_leaky.drop(columns=['LEAK_is_declining_label'])
del Xl_train, Xl_test, yl_train, yl_test, leaky
SAFE_FEATURES = ['feb_impressions']
assert 'LEAK_is_declining_label' not in X_leaky.columns
assert 'is_declining_label' not in SAFE_FEATURES
print('Leak removed. Safe feature columns:', SAFE_FEATURES)
print(f'Honest reference accuracy to retain: {honest_score:.4f}')


## 6. One named limitation

**Unbalanced panel and uneven measurement coverage.** Clients enter the panel at different dates, and GA4 availability is not universal. A content item with fewer observed March reporting days may reflect tracking/history coverage rather than genuinely low search activity. This feature frame therefore describes observed March search signals; it is not a complete, equally observed census of every content item. Also, a one-month change is not proof of lasting decline, and observational data cannot prove that refreshing content causes recovery.

## 7. Self-check

- [ ] Contract covers row meaning, source tables, time window, decision/proxy, and one deliberate exclusion.
- [ ] Exactly three verification queries were run; their real outputs are visible.
- [ ] Availability query uses `ga4_data_available IS TRUE`.
- [ ] Five March features are built, each with an “available when?” explanation.
- [ ] The intentional label leak is shown, then deleted from the feature set.
- [ ] Notebook runs top to bottom and the executed version is saved to GitHub.
- [ ] No token, client names, URLs, or private queries are committed.

**Important:** query counts and model scores are produced by your authenticated run. Do not manually invent them. After running in Colab, use *File → Save a copy in GitHub* to commit the notebook with outputs.